<a href="https://colab.research.google.com/github/Jef-H/super_classy_fields/blob/develop/super_classy_fields_FP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Supervised Learning Classification: Field or Not a Field?

## Project Overview

This project aims to train a model to classify satellite images into two categories:
- **Field** (most likely corn, soy, or alfalfa)
- **Not a Field**

## Data Sources

### Field Images
- Screenshots of cornfields were taken and cropped to match the Kaggle dataset size (64x64 pixels).
- Data obtained from **ESRI layers at USGS** via [Earth Explorer](https://earthexplorer.usgs.gov/).

### Other Category Images
- Non-field images were sourced from a Kaggle dataset:  
  [Satellite Image Classification Dataset](https://www.kaggle.com/datasets/mahmoudreda55/satellite-image-classification/data).  
  Categories include:
  - Cloudy
  - Desert
  - Green
  - Water

## Goals
The objective is to see how well the model can distinguish between "Field" and "Not a Field" categories using these datasets.

---

Let's see how it goes!



In [2]:
import seaborn as sns
import matplotlib.pyplot as plt
import os
from PIL import Image
import pandas as pd
import zipfile
import requests
import time
import tempfile
import shutil

In [3]:
def download_png_files_from_github(github_url, output_dir):
    """
    Downloads all PNG files from a specified GitHub repository directory,
    clearing the destination directory before downloading.

    Args:
        github_url (str): The GitHub URL of the repository directory.
        output_dir (str): Directory to save the downloaded PNG files.
    """
    # Ensure the output directory exists and clean it
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)  # Remove all files in the directory
    os.makedirs(output_dir, exist_ok=True)
    print(f"Cleaned and prepared output directory: {output_dir}")

    # Extract the repository name and directory from the GitHub URL
    repo_url = github_url.split("https://github.com/")[1]
    repo_name, tree_path = repo_url.split("/tree/")

    # Correctly format the path for the API (removing 'tree' and using branch name)
    branch_name = tree_path.split('/')[0]  # Assuming branch name is the first part of the path
    dir_path = '/'.join(tree_path.split('/')[1:])  # Directory path is the rest

    # GitHub API URL to list the contents of the directory
    api_url = f"https://api.github.com/repos/{repo_name}/contents/{dir_path}?ref={branch_name}"

    # Send a request to the GitHub API
    response = requests.get(api_url)
    if response.status_code != 200:
        print(f"Failed to fetch contents from {api_url}. Status code: {response.status_code}")
        return

    files = response.json()
    png_count = 0  # Counter for PNG files

    # Iterate through each file in the directory
    for file in files:
        # Check if the file is a PNG
        if file['name'].endswith('.png'):
            file_url = file['download_url']
            file_name = file['name']
            file_path = os.path.join(output_dir, file_name)

            # Download and save the PNG file
            file_response = requests.get(file_url)
            if file_response.status_code == 200:
                with open(file_path, 'wb') as f:
                    f.write(file_response.content)
                png_count += 1  # Increment the PNG counter
            else:
                print(f"Failed to download {file_name}")

    print(f"Downloaded {png_count} PNG files to {output_dir}")

# Example usage
github_url = "https://github.com/Jef-H/supervised_fields/tree/develop/crop_field_images"
output_directory = "downloaded_png_files/"

download_png_files_from_github(github_url, output_directory)


Cleaned and prepared output directory: downloaded_png_files/
Downloaded 125 PNG files to downloaded_png_files/


In [4]:
def cut_images_to_fragments(input_dir, output_dir, fragment_size=(64, 64)):
    """
    Cuts all PNG images in the input directory into fragments of the given size
    and saves them to the output directory. Excess parts of the images are discarded.

    Args:
        input_dir (str): Directory containing PNG images.
        output_dir (str): Directory to save the image fragments.
        fragment_size (tuple): Size of each fragment (width, height).
    """
    start_time = time.time()  # Start the timer

    # Clear the output directory before writing new fragments
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)  # Remove the existing directory and its contents
    os.makedirs(output_dir, exist_ok=True)  # Create a fresh directory
    print(f"Ensured output directory is cleared and exists: {output_dir}")

    total_fragment_count = 0  # Counter for all fragments across all images

    # Iterate through each file in the input directory
    for filename in os.listdir(input_dir):
        if filename.endswith('.png'):
            file_path = os.path.join(input_dir, filename)
            with Image.open(file_path) as img:
                img_width, img_height = img.size

                # Calculate the number of fragments in each dimension
                num_fragments_x = img_width // fragment_size[0]
                num_fragments_y = img_height // fragment_size[1]

                total_fragments = 0

                # Generate and save each fragment
                for i in range(num_fragments_x):
                    for j in range(num_fragments_y):
                        left = i * fragment_size[0]
                        upper = j * fragment_size[1]
                        right = left + fragment_size[0]
                        lower = upper + fragment_size[1]

                        fragment = img.crop((left, upper, right, lower))

                        fragment_filename = f"{os.path.splitext(filename)[0]}_{i}_{j}.png"
                        fragment_path = os.path.join(output_dir, fragment_filename)
                        fragment.save(fragment_path)

                        total_fragments += 1
                        total_fragment_count += 1

    print(f"Total fragments created for all images: {total_fragment_count}")

    # End the timer and print the elapsed time
    elapsed_time = time.time() - start_time
    print(f"Processing completed in {elapsed_time:.2f} seconds.")

# Example usage
github_url = "https://github.com/Jef-H/supervised_fields/tree/develop/crop_field_images"
output_directory = "crop_fragments/"

# Create temporary directory outside the 'with' context to avoid deletion
temp_dir = tempfile.mkdtemp()
print(f"Temporary directory created: {temp_dir}")
input_directory = "downloaded_png_files/"

cut_images_to_fragments(input_directory, output_directory)


Temporary directory created: /tmp/tmpne1_578a
Ensured output directory is cleared and exists: crop_fragments/
Total fragments created for all images: 5343
Processing completed in 10.75 seconds.


In [5]:
# URL of the ZIP file on GitHub
zip_url = 'https://github.com/Jef-H/supervised_fields/blob/develop/Sat_img_class.zip?raw=true'

# Define the directory to save the extracted files
extract_dir = 'satellite_images'

# Step 1: Download the ZIP file
response = requests.get(zip_url)
zip_file_path = 'Sat_img_class.zip'

# Save the file
with open(zip_file_path, 'wb') as f:
    f.write(response.content)

# Step 2: Extract the ZIP file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f'ZIP file extracted to {extract_dir}')

# Step 3: Optional - Display the extracted files
extracted_files = os.listdir(extract_dir)
print(f'Files extracted: {extracted_files}')

# You can also process the images, e.g., using PIL or other libraries


ZIP file extracted to satellite_images
Files extracted: ['data']


In [11]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, Input

# Set dataset paths
field_images_path = "crop_fragments/"  # Path for field images
not_field_images_paths = [
    "satellite_images/data/cloudy",
    "satellite_images/data/desert",
    "satellite_images/data/green_area",
    "satellite_images/data/water"
]

# Parameters
image_size = (64, 64)  # Image dimensions for resizing

# Helper function to load images
def load_images_from_directory(directory, label):
    """Loads images from a directory and assigns a label."""
    images = []
    labels = []
    for filename in os.listdir(directory):
        filepath = os.path.join(directory, filename)
        try:
            img = tf.keras.utils.load_img(filepath, target_size=image_size)
            img_array = tf.keras.utils.img_to_array(img) / 255.0  # Normalize
            images.append(img_array)
            labels.append(label)
        except Exception as e:
            print(f"Error loading image {filename}: {e}")
    return np.array(images), np.array(labels)

# Load field data
field_images, field_labels = load_images_from_directory(field_images_path, label="field")

# Load not-field data from multiple directories
not_field_images = []
not_field_labels = []
for path in not_field_images_paths:
    images, labels = load_images_from_directory(path, label="not_field")
    not_field_images.append(images)
    not_field_labels.append(labels)

try:
    not_field_images = np.concatenate(not_field_images, axis=0)
    not_field_labels = np.concatenate(not_field_labels, axis=0)
except ValueError:
    print("Error: No images loaded for the 'not_field' category.")

# Combine data
try:
    X = np.concatenate([field_images, not_field_images], axis=0)
    y = np.concatenate([field_labels, not_field_labels], axis=0)
except ValueError:
    print("Error: Ensure both 'field' and 'not_field' data are loaded correctly.")
    X, y = None, None

if X is not None and y is not None:
    # Encode labels
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(y)  # Convert "field"/"not_field" to 0/1

    # Split data
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

    # Data augmentation for CNNs
    data_gen = ImageDataGenerator(
        rotation_range=20,
        width_shift_range=0.1,
        height_shift_range=0.1,
        zoom_range=0.2,
        horizontal_flip=True
    )

    data_gen.fit(X_train)

    print("Data preparation complete:")
    print(f"Training data: {X_train.shape}, {y_train.shape}")
    print(f"Validation data: {X_val.shape}, {y_val.shape}")
    print(f"Test data: {X_test.shape}, {y_test.shape}")
else:
    print("Data preparation failed. Check the dataset and paths.")



Data preparation complete:
Training data: (7681, 64, 64, 3), (7681,)
Validation data: (1646, 64, 64, 3), (1646,)
Test data: (1647, 64, 64, 3), (1647,)


In [12]:

# Build CNN model
model = Sequential([
    Input(shape=(64, 64, 3)),
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    BatchNormalization(),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    BatchNormalization(),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    BatchNormalization(),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Summary
model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)                    │ (None, 62, 62, 32)          │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_3 (MaxPooling2D)       │ (None, 31, 31, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (None, 31, 31, 32)          │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_4 (Conv2D)                    │ (None, 29, 29, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_4 (MaxPooling2D)       │ (None, 14, 14, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_4                │ (None, 14, 14, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_5 (Conv2D)                    │ (None, 12, 12, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_5 (MaxPooling2D)       │ (None, 6, 6, 128)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_5                │ (None, 6, 6, 128)           │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_1 (Flatten)                  │ (None, 4608)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 128)                 │         589,952 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 1)                   │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 684,225 (2.61 MB)

 Trainable params: 683,777 (2.61 MB)

 Non-trainable params: 448 (1.75 KB)

# Reflection Questions

Answer the following questions about your supervised learning classification project. Write 1-3 lines per answer in separate Markdown cells.

---

### 1. Which method did you like the most?
*(Your answer here)*

---

### 2. Which method did you like the least?
*(Your answer here)*

---

### 3. How did you score these supervised models?  
*(Your answer here)*

---

### 4. Did the output align with your geologic understanding?  
*(Your answer here)*

---

### 5. Did you hyperparameter tune? Why or why not?  
*(Your answer here)*

---

### 6. How did you split your data? And why does that make sense for this dataset?  
*(Your answer here)*

---

### 7. What did you want to learn more about?  
*(Your answer here)*

---

### 8. Did you pre-process your data?  
*(Your answer here)*
